In [0]:
events=spark.table("spotify_silver.fact_listening_events")
users=spark.table("spotify_silver.dim_users")

events.createOrReplaceTempView("fact_listening_events")
users.createOrReplaceTempView("dim_users")

In [0]:
%sql
CREATE OR REPLACE TABLE spotify_gold.daily_listening_trend AS

SELECT

DATE(timestamp) AS listening_date,

COUNT(*) AS total_events,

COUNT(DISTINCT user_id) AS active_users,

SUM(listen_duration_ms) AS total_listening_time_ms,

ROUND(
AVG(listen_duration_ms),
2
) AS avg_listening_duration_ms

FROM fact_listening_events

GROUP BY DATE(timestamp)

ORDER BY listening_date;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM spotify_gold.daily_listening_trend
LIMIT 10;

listening_date,total_events,active_users,total_listening_time_ms,avg_listening_duration_ms
2023-01-01,91,91,14497664,159314.99
2023-01-02,82,82,14018380,170955.85
2023-01-03,88,88,13282046,150932.34
2023-01-04,76,75,13096637,172324.17
2023-01-05,86,86,15268438,177539.98
2023-01-06,86,86,14439911,167905.94
2023-01-07,96,94,14366700,149653.13
2023-01-08,87,87,15557056,178816.74
2023-01-09,100,99,15932628,159326.28
2023-01-10,87,86,14557884,167332.0


In [0]:
%sql
CREATE OR REPLACE TABLE spotify_gold.monthly_listening_trend AS

SELECT

YEAR(timestamp) AS year,

MONTH(timestamp) AS month,

COUNT(*) AS total_events,

COUNT(DISTINCT user_id) AS active_users,

SUM(listen_duration_ms) AS total_listening_time_ms

FROM fact_listening_events

GROUP BY
YEAR(timestamp),
MONTH(timestamp)

ORDER BY
year,
month;

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from spotify_gold.monthly_listening_trend
limit 10;

year,month,total_events,active_users,total_listening_time_ms
2023,1,2818,2173,468342454
2023,2,2476,1956,397963749
2023,3,2799,2153,459906884
2023,4,2733,2115,450569093
2023,5,2842,2168,469916012
2023,6,2731,2118,455601684
2023,7,2833,2183,461594634
2023,8,2877,2206,476254926
2023,9,2684,2080,441437790
2023,10,2797,2123,461555020


**EXECUTIVE KPIs**

In [0]:
%sql
CREATE OR REPLACE TABLE spotify_gold.executive_kpis AS

SELECT
    (SELECT COUNT(*) FROM spotify_silver.dim_users) AS total_users,

    (SELECT COUNT(*) FROM spotify_silver.dim_tracks) AS total_tracks,

    (SELECT COUNT(*) FROM spotify_silver.dim_artists) AS total_artists,

    (SELECT COUNT(*) FROM spotify_silver.dim_albums) AS total_albums,

    (SELECT COUNT(*) FROM spotify_silver.fact_listening_events) AS total_listening_events,

    ROUND(
        (SELECT SUM(listen_duration_ms)
         FROM spotify_silver.fact_listening_events) / 3600000,
        2
    ) AS total_listening_hours,

    ROUND(
        (SELECT AVG(listen_duration_ms)
         FROM spotify_silver.fact_listening_events),
        2
    ) AS avg_listening_duration_ms,

    ROUND(
        (
            SELECT 100 * AVG(CASE WHEN completed THEN 1 ELSE 0 END)
            FROM spotify_silver.fact_listening_events
        ),
        2
    ) AS completion_rate,

    ROUND(
        (
            SELECT 100 * AVG(CASE WHEN skipped THEN 1 ELSE 0 END)
            FROM spotify_silver.fact_listening_events
        ),
        2
    ) AS skip_rate;

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from spotify_gold.executive_kpis;

total_users,total_tracks,total_artists,total_albums,total_listening_events,total_listening_hours,avg_listening_duration_ms,completion_rate,skip_rate
5000,8772,2547,5314,100000,4575.1,164703.65,80.28,19.82
